In [145]:
import torch
import torch.nn as nn
import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


In [146]:
#data preprocessing

bc = datasets.load_breast_cancer()

X,y = bc.data,bc.target
n_samples,n_features = X.shape
n_samples,n_features

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.from_numpy(X_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
y_test = torch.from_numpy(y_test.astype(np.float32))

y_train = y_train.view(y_train.shape[0],1)
y_test = y_test.view(y_test.shape[0],1)


In [147]:
class LogisticRegression(nn.Module):

    def __init__(self, n_input):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(n_input, 1)

    def forward(self, x):
            y_pred = torch.sigmoid(self.linear(x))
            return y_pred

In [148]:
model = LogisticRegression(n_features)

In [149]:
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

In [150]:
n_epochs = 1000

for epoch in range(n_epochs):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if (epoch+1) % 100 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.4f}')

Epoch 100/1000, Loss: 0.6280
Epoch 200/1000, Loss: 0.5104
Epoch 300/1000, Loss: 0.4393
Epoch 400/1000, Loss: 0.3918
Epoch 500/1000, Loss: 0.3575
Epoch 600/1000, Loss: 0.3313
Epoch 700/1000, Loss: 0.3106
Epoch 800/1000, Loss: 0.2936
Epoch 900/1000, Loss: 0.2795
Epoch 1000/1000, Loss: 0.2674


In [151]:
with torch.no_grad():
    y_pred = model(X_test)
    y_pred_cls = y_pred.round()
    acc = y_pred_cls.eq(y_test).sum() / float(y_test.shape[0])
    print(f'accuracy: {acc.item():.4f}')

accuracy: 0.9737
